In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("IMDB Dataset.csv")

In [5]:
df.shape

(50000, 2)

In [6]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [8]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [9]:
df.drop_duplicates(inplace = True)

In [10]:
df.shape

(49582, 2)

## Pre-processing

### 1. Converting to lowercase

In [11]:
df["review"] = df["review"].str.lower()

### 2. Removing the URLs

In [14]:
import re

# sample_text = "abc is the word, abc" # abc => xyz

# new_text = re.sub("abc", "xyz", sample_text)

In [15]:
def remove_urls(text):
    text = re.sub(r"http\S+" , "", text)  # (pattern, repl, string) eg - https://www.google.com
    return text
    
df["review"] = df["review"].apply(remove_urls)

### 3. Removing punctuations

In [16]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "", text) # A-Z a-z 0-9 \s
    return text

df["review"] = df["review"].apply(remove_punctuations)

In [17]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 4. Removing HTML

In [18]:
def remove_html(text):
    text = re.sub(r"<.*?>" , "", text)
    return text

df["review"] = df["review"].apply(remove_html)

### 5. Removing the Stopwords

In [19]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [20]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [21]:
# sample_text = "I like coding in python!"
# tokens = word_tokenize(sample_text)

In [23]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [24]:
df.head()

,review,sentiment
0,e revewers nte wtchg 1 oz epoe hooke ...,positive
1,wderful ltle prducti br br filming technique...,positive
2,hugh h werful w pen me h ummer weeken n...,positive
3,bcy fly e boy jke hk zobe cloe pn f...,negative
4,peer me love i vu unng film wch mr mei ...,positive


### 6. Stemming

In [25]:
# running -> run
# played -> play
# PorterStemming

from nltk.stem import PorterStemmer

In [28]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [29]:
df.head()

,review,sentiment
0,e revew nte wtchg 1 oz epo hook rght exctl hpp...,positive
1,wder ltle prducti br br film techniqu unssum l...,positive
2,hugh h wer w pen me h ummer weeken ng n r cne ...,positive
3,bci fli e boy jke hk zobe cloe pn fghg ebr br ...,negative
4,peer me love i vu unng film wch mr mei fer u v...,positive


### 7. Encoding

In [30]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [31]:
y = df["sentiment"]

In [32]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int32

### 8. Vectorization

In [33]:
df.head()

,review,sentiment
0,e revew nte wtchg 1 oz epo hook rght exctl hpp...,1
1,wder ltle prducti br br film techniqu unssum l...,1
2,hugh h wer w pen me h ummer weeken ng n r cne ...,1
3,bci fli e boy jke hk zobe cloe pn fghg ebr br ...,0
4,peer me love i vu unng film wch mr mei fer u v...,1


In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [37]:
X

<49582x5000 sparse matrix of type '<class 'numpy.float64'>'
	with 3316102 stored elements in Compressed Sparse Row format>

## Dataset & Data Loaders

In [38]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state=42
)

In [39]:
X_train.shape

(39665, 5000)

In [40]:
X_test.shape

(9917, 5000)

In [42]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [43]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [44]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [45]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

## Build our RNN

In [46]:
import torch.nn as nn
import torch.optim as optim

In [47]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [48]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the RNN

In [49]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.28223058581352234
epoch = 2/10 and loss = 0.27307069301605225
epoch = 3/10 and loss = 0.15065760910511017
epoch = 4/10 and loss = 0.33938226103782654
epoch = 5/10 and loss = 0.4597756564617157
epoch = 6/10 and loss = 0.3453015387058258
epoch = 7/10 and loss = 0.24688473343849182
epoch = 8/10 and loss = 0.31741049885749817
epoch = 9/10 and loss = 0.29568061232566833
epoch = 10/10 and loss = 0.25269758701324463


In [50]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 82.49470606030049
